# 03 张量运算和广播机制

本节目标：掌握逐元素运算、矩阵乘法、广播、拼接、降维、范数，并重点理解广播为什么成立。

## 1. 广播规则

两个张量做逐元素运算时，PyTorch 会从最后一个维度开始比较 shape：

- 两个维度相等，可以运算。
- 其中一个维度是 `1`，可以广播成另一个维度。
- 如果既不相等，也没有 `1`，就不能广播。

广播不会真的手动复制很多份数据，它只是让我们用更简洁的方式写逐元素运算。

In [ ]:
import torch

torch.set_printoptions(precision=2)


def show(name, value):
    print(f"{name}:")
    print(value)
    print("shape:", tuple(value.shape))
    print()


def shape_diagram(title, left, op, right, result):
    print(title)
    print(f"  left   {tuple(left.shape)}")
    print(f"  {op}")
    print(f"  right  {tuple(right.shape)}")
    print("  ----------------")
    print(f"  result {tuple(result.shape)}")
    print()

## 2. 逐元素运算

shape 完全相同的两个张量，可以直接做加、减、乘、除等逐元素运算。

In [ ]:
x = torch.tensor([1, 2, 3])
y = torch.tensor([10, 20, 30])

show("x", x)
show("y", y)
show("x + y", x + y)
show("x * y", x * y)

## 3. 广播例子 1：列向量 + 行向量

`a` 的 shape 是 `(3, 1)`，`b` 的 shape 是 `(1, 4)`。

从最后一维比较：

- 最后一维：`1` 和 `4`，`1` 可以广播成 `4`
- 倒数第二维：`3` 和 `1`，`1` 可以广播成 `3`

所以结果 shape 是 `(3, 4)`。

In [ ]:
a = torch.arange(3).reshape(3, 1)
b = torch.arange(4).reshape(1, 4)
c = a + b

show("a", a)
show("b", b)
show("a + b", c)
shape_diagram("广播例子 1", a, "+", b, c)

## 4. 广播例子 2：矩阵 + 向量

`matrix` 的 shape 是 `(2, 3)`，`bias` 的 shape 是 `(3,)`。

比较时可以把 `(3,)` 看成 `(1, 3)`：

- 最后一维：`3` 和 `3`，相等
- 倒数第二维：`2` 和 `1`，`1` 可以广播成 `2`

所以每一行都会加上同一个 `bias`。

In [ ]:
matrix = torch.arange(6).reshape(2, 3)
bias = torch.tensor([100, 200, 300])
result = matrix + bias

show("matrix", matrix)
show("bias", bias)
show("matrix + bias", result)
shape_diagram("广播例子 2", matrix, "+", bias, result)

## 5. 例子 3：矩阵乘法 shape

矩阵乘法不是广播，而是线性代数运算。

`A` 的 shape 是 `(2, 3)`，`B` 的 shape 是 `(3, 2)`。中间维度 `3` 对齐，所以输出 shape 是 `(2, 2)`。

In [ ]:
A = torch.arange(1, 7).reshape(2, 3)
B = torch.arange(1, 7).reshape(3, 2)
C = A @ B

show("A", A)
show("B", B)
show("A @ B", C)
shape_diagram("矩阵乘法例子", A, "@", B, C)

## 6. 拼接、降维和范数

- 拼接：把多个张量沿某个维度接起来。
- 降维：沿某个维度求和或求均值，维度数量可能减少。
- 范数：衡量向量或矩阵大小，例如 L2 范数。

In [ ]:
X = torch.arange(6, dtype=torch.float32).reshape(2, 3)
Y = torch.ones((2, 3))

cat_dim0 = torch.cat([X, Y], dim=0)
cat_dim1 = torch.cat([X, Y], dim=1)
sum_dim0 = X.sum(dim=0)
mean_dim1 = X.mean(dim=1)
l2_norm = torch.linalg.vector_norm(X)

show("X", X)
show("Y", Y)
show("按 dim=0 拼接", cat_dim0)
show("按 dim=1 拼接", cat_dim1)
show("X.sum(dim=0)", sum_dim0)
show("X.mean(dim=1)", mean_dim1)
print("X 的 L2 范数:", l2_norm)
print("L2 范数 shape:", tuple(l2_norm.shape))

## 小结

广播成立的关键：从最后一个维度开始看，每一维要么相等，要么其中一个是 `1`。做张量运算前先写出输入 shape，很多错误都能提前发现。